⚠️ **Security note:** This notebook uses `dbutils.secrets` to retrieve the ADLS storage key.
Never hardcode credentials. The secret scope `adls` must be configured before running:
```
databricks secrets create-scope --scope adls
databricks secrets put-secret --scope adls --key storage-account-key
```

# Gold Layer — Road Network ETL Pipeline
Reads Silver Delta tables from ADLS, builds star schema, writes Gold Delta tables.

**Inputs (Silver Delta):**
- `silver/osm/road_segments_silver`
- `silver/waka_kotahi/tms_counts_silver`
- `silver/waka_kotahi/tms_monitoring_sites_silver`

**Outputs (Gold Delta):**
- `gold/dim_road_segment`
- `gold/dim_location`
- `gold/dim_time`
- `gold/fact_traffic_counts`

In [0]:
# ── ADLS Configuration ───────────────────────────────────
STORAGE_ACCOUNT = "nzetlpipeline"
CONTAINER       = "medallion"
KEY             = dbutils.secrets.get(scope="adls", key="storage-account-key")

spark.conf.set(
    f"fs.azure.account.key.{STORAGE_ACCOUNT}.dfs.core.windows.net",
    KEY
)

BASE_PATH = f"abfss://{CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net"

# ── Date parameter — inherited from job, not used directly in gold ───────────
# Gold reads from silver Delta tables which are date-independent
dbutils.widgets.text("bronze_date", "2026-03-20", "Bronze Date (YYYY-MM-DD)")

# Silver input paths
SILVER_SEGMENTS = f"{BASE_PATH}/silver/osm/road_segments_silver"
SILVER_COUNTS   = f"{BASE_PATH}/silver/waka_kotahi/tms_counts_silver"
SILVER_SITES    = f"{BASE_PATH}/silver/waka_kotahi/tms_monitoring_sites_silver"

# Gold output paths
GOLD_SEGMENT    = f"{BASE_PATH}/gold/dim_road_segment"
GOLD_LOCATION   = f"{BASE_PATH}/gold/dim_location"
GOLD_TIME       = f"{BASE_PATH}/gold/dim_time"
GOLD_FACT       = f"{BASE_PATH}/gold/fact_traffic_counts"

print(f"Config loaded.")

Config loaded.


## 1. Load Silver inputs

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import (
    LongType, IntegerType, ShortType, ByteType,
    BooleanType, StringType, DoubleType
)

segments_silver = spark.read.format("delta").load(SILVER_SEGMENTS)
counts_silver   = spark.read.format("delta").load(SILVER_COUNTS)
sites_silver    = spark.read.format("delta").load(SILVER_SITES)

print(f"road_segments_silver : {segments_silver.count():,} rows")
print(f"tms_counts_silver    : {counts_silver.count():,} rows")
print(f"tms_sites_silver     : {sites_silver.count():,} rows")

road_segments_silver : 25,366 rows
tms_counts_silver    : 389,791 rows
tms_sites_silver     : 233 rows


## 2. Spatial Join — Sites to Nearest OSM Road Segment
Uses a pandas UDF to run geopandas sjoin_nearest on the driver.
Safe for this data size (246 sites × 25k segments).

In [0]:
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
from pyspark.sql.types import StructType, StructField

# Collect segments — use the row index as the future segment_id
segments_pd = segments_silver.select("osmid", "geometry").toPandas()
segments_pd["segment_id"] = segments_pd.index  # row index = future surrogate key

sites_pd = sites_silver.select("siteReference", "easting", "northing").toPandas()

# Build GeoDataFrames
segments_gdf = gpd.GeoDataFrame(
    segments_pd,
    geometry=gpd.GeoSeries.from_wkb(segments_pd["geometry"], crs="EPSG:2193"),
    crs="EPSG:2193"
)

sites_gdf = gpd.GeoDataFrame(
    sites_pd,
    geometry=gpd.points_from_xy(sites_pd["easting"], sites_pd["northing"]),
    crs="EPSG:2193"
)

# Spatial join
sites_joined = gpd.sjoin_nearest(
    sites_gdf,
    segments_gdf[["segment_id", "geometry"]],
    how="left",
    distance_col="distance_to_road_m"
)

# Keep closest match per site
sites_joined = (
    sites_joined
    .sort_values("distance_to_road_m")
    .drop_duplicates(subset=["siteReference"])
    .reset_index(drop=True)
)

print(f"Sites joined: {len(sites_joined):,}")
print(f"Max distance : {sites_joined['distance_to_road_m'].max():.1f} m")
print(f"Median distance : {sites_joined['distance_to_road_m'].median():.1f} m")

far_sites = sites_joined[sites_joined["distance_to_road_m"] > 500]
print(f"Sites > 500m from road: {len(far_sites)}")

# Convert back to Spark
sites_to_osm = spark.createDataFrame(
    sites_joined[["siteReference", "segment_id", "distance_to_road_m"]]
)

Sites joined: 233
Max distance : 205573.9 m
Median distance : 2201.3 m
Sites > 500m from road: 125


## 3. dim_road_segment

In [0]:
# Assign surrogate key using monotonically_increasing_id
# then zipWithIndex for stable 0-based integers
from pyspark.sql.window import Window

dim_road_segment = (
    segments_silver
    .drop("geometry")
    .withColumn(
        "segment_id",
        (F.row_number().over(Window.orderBy(F.monotonically_increasing_id())) - 1).cast(LongType())
    )
    .select("segment_id", "osmid", "name", "road_class", "maxspeed_kmh", "oneway", "length", "lanes")
)

print(f"dim_road_segment: {dim_road_segment.count():,} rows")
dim_road_segment.show(5, truncate=False)

/databricks/spark/python/pyspark/sql/connect/expressions.py:1134: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


dim_road_segment: 25,366 rows
+----------+--------------------------------+------------------------+----------+------------+------+------------------+-----+
|segment_id|osmid                           |name                    |road_class|maxspeed_kmh|oneway|length            |lanes|
+----------+--------------------------------+------------------------+----------+------------+------+------------------+-----+
|0         |[1133615064, 5365476, 597540814]|Durey Road              |collector |NULL        |true  |56.928545045026226|1    |
|1         |488963306                       |Sister Cities Roundabout|arterial  |50          |true  |29.95627054336464 |2    |
|2         |4243735                         |Sister Cities Roundabout|arterial  |50          |true  |4.084168735002902 |2    |
|3         |1035329059                      |Sister Cities Roundabout|arterial  |50          |true  |4.7344343679711836|2    |
|4         |837770558                       |Sister Cities Roundabout|arterial  |

In [0]:
(
    dim_road_segment
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(GOLD_SEGMENT)
)
print(f"Saved: {GOLD_SEGMENT}")

/databricks/spark/python/pyspark/sql/connect/expressions.py:1134: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


Saved: abfss://medallion@nzetlpipeline.dfs.core.windows.net/gold/dim_road_segment


## 4. dim_location

In [0]:
# Build osmid → segment_id lookup from dim_road_segment
osmid_to_seg = dim_road_segment.select("osmid", "segment_id")
# MAX_DISTANCE_M = 5000 # meters (we put a null seg_id for roads beyond this distance)

# Join sites_silver with spatial join results and segment lookup
dim_location = (
    sites_silver
    .join(sites_to_osm, on="siteReference", how="left")
    .select(
        "siteReference",
        "region",
        "site_description",
        "easting",
        "northing",
        "segment_id",
        "distance_to_road_m",
    )
    # # Null out segment_id for sites too far from any road
    # .withColumn(
    #     "segment_id",
    #     F.when(
    #         F.col("distance_to_road_m") > MAX_DISTANCE_M, None
    #     ).otherwise(F.col("segment_id"))
    # )
    # Assign surrogate site_id
    .withColumn(
        "site_id",
        F.row_number().over(Window.orderBy("siteReference")).cast(LongType()) - 1
    )
    .select(
        "site_id",
        "siteReference",
        "region",
        "site_description",
        "easting",
        "northing",
        "segment_id",
        "distance_to_road_m",
    )
)

print(f"dim_location: {dim_location.count():,} rows")
print(f"Sites with segment_id : {dim_location.filter(F.col('segment_id').isNotNull()).count():,}")
# print(f"Sites without (far)   : {dim_location.filter(F.col('segment_id').isNull()).count():,}")
dim_location.show(5, truncate=False)

/databricks/spark/python/pyspark/sql/connect/expressions.py:1134: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


dim_location: 233 rows
Sites with segment_id : 233
+-------+-------------+---------------+---------------------------------------------------+------------+------------+----------+------------------+
|site_id|siteReference|region         |site_description                                   |easting     |northing    |segment_id|distance_to_road_m|
+-------+-------------+---------------+---------------------------------------------------+------------+------------+----------+------------------+
|0      |00700002     |11 - Canterbury|Waipara Straight - 2km Nth-West of Waipara Junction|1579007.7142|5232576.5936|18530     |42819.38040691687 |
|1      |00700014     |11 - Canterbury|Waikari - Nth-west of Township                     |1576013.5258|5242477.3285|18530     |52175.2823093098  |
|2      |00700041     |11 - Canterbury|Culverden - At School Creek Bridge                 |1587746.444 |5264561.5904|18530     |75929.7835590591  |
|3      |00700044     |11 - Canterbury|Red Post Corner - Sth 

In [0]:
(
    dim_location
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(GOLD_LOCATION)
)
print(f"Saved: {GOLD_LOCATION}")

/databricks/spark/python/pyspark/sql/connect/expressions.py:1134: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


Saved: abfss://medallion@nzetlpipeline.dfs.core.windows.net/gold/dim_location


## 5. dim_time

In [0]:
# Generate date range from actual counts data
date_bounds = counts_silver.agg(
    F.min("date").alias("min_date"),
    F.max("date").alias("max_date")
).collect()[0]

print(f"Date range: {date_bounds.min_date} → {date_bounds.max_date}")

# Generate all dates in range using sequence
dim_time = (
    spark.sql(f"""
        SELECT explode(sequence(
            date('{date_bounds.min_date}'),
            date('{date_bounds.max_date}'),
            interval 1 day
        )) AS date
    """)
    .withColumn("date_id",      F.date_format("date", "yyyyMMdd").cast(IntegerType()))
    .withColumn("year",         F.year("date").cast(ShortType()))
    .withColumn("month",        F.month("date").cast(ByteType()))
    .withColumn("month_name",   F.date_format("date", "MMMM"))
    .withColumn("day_of_month", F.dayofmonth("date").cast(ByteType()))
    .withColumn("day_of_week",  (F.dayofweek("date") - 2).cast(ByteType()))  # 0=Mon
    .withColumn("day_name",     F.date_format("date", "EEEE"))
    .withColumn("week_of_year", F.weekofyear("date").cast(ByteType()))
    .withColumn("quarter",      F.quarter("date").cast(ByteType()))
    .withColumn("is_weekend",   F.dayofweek("date").isin([1, 7]))  # 1=Sun, 7=Sat
    # NZ public holidays — fixed dates (variable dates like Easter excluded)
    .withColumn(
        "is_nz_holiday",
        F.date_format("date", "MM-dd").isin([
            "01-01",  # New Year's Day
            "01-02",  # Day after New Year's
            "02-06",  # Waitangi Day
            "04-25",  # ANZAC Day
            "06-03",  # King's Birthday (approx — variable)
            "10-27",  # Labour Day (approx — variable)
            "12-25",  # Christmas Day
            "12-26",  # Boxing Day
        ])
    )
)

print(f"dim_time: {dim_time.count():,} rows")
dim_time.show(5)

Date range: 2018-01-01 → 2023-09-03
dim_time: 2,072 rows
+----------+--------+----+-----+----------+------------+-----------+---------+------------+-------+----------+-------------+
|      date| date_id|year|month|month_name|day_of_month|day_of_week| day_name|week_of_year|quarter|is_weekend|is_nz_holiday|
+----------+--------+----+-----+----------+------------+-----------+---------+------------+-------+----------+-------------+
|2018-01-01|20180101|2018|    1|   January|           1|          0|   Monday|           1|      1|     false|         true|
|2018-01-02|20180102|2018|    1|   January|           2|          1|  Tuesday|           1|      1|     false|         true|
|2018-01-03|20180103|2018|    1|   January|           3|          2|Wednesday|           1|      1|     false|        false|
|2018-01-04|20180104|2018|    1|   January|           4|          3| Thursday|           1|      1|     false|        false|
|2018-01-05|20180105|2018|    1|   January|           5|          4|

In [0]:
(
    dim_time
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(GOLD_TIME)
)
print(f"Saved: {GOLD_TIME}")

Saved: abfss://medallion@nzetlpipeline.dfs.core.windows.net/gold/dim_time


## 6. fact_traffic_counts

In [0]:
# Build lookup maps
# site_id lookup: siteReference → site_id
site_lookup = dim_location.select("siteReference", "site_id", "segment_id")

# date_id lookup: date → date_id
date_lookup = dim_time.select("date", "date_id")

fact_traffic_counts = (
    counts_silver
    # Attach site_id and segment_id
    .join(site_lookup, on="siteReference", how="inner")
    # Attach date_id
    .join(date_lookup, on="date", how="inner")
    # Final columns only
    .select(
        "date_id",
        "site_id",
        "segment_id",
        "vehicle_class",
        F.col("daily_count").cast(LongType()),
    )
)

print(f"fact_traffic_counts: {fact_traffic_counts.count():,} rows")
fact_traffic_counts.show(5)

/databricks/spark/python/pyspark/sql/connect/expressions.py:1134: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


fact_traffic_counts: 353,126 rows
+--------+-------+----------+-------------+-----------+
| date_id|site_id|segment_id|vehicle_class|daily_count|
+--------+-------+----------+-------------+-----------+
|20200103|      0|     18530|        Heavy|        747|
|20210805|      1|     18530|        Heavy|        758|
|20190124|      2|     18530|        Heavy|        524|
|20230304|      3|        68|        Heavy|        601|
|20190724|      4|     18530|        Heavy|        354|
+--------+-------+----------+-------------+-----------+
only showing top 5 rows


In [0]:
(
    fact_traffic_counts
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(GOLD_FACT)
)
print(f"Saved: {GOLD_FACT}")

/databricks/spark/python/pyspark/sql/connect/expressions.py:1134: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


Saved: abfss://medallion@nzetlpipeline.dfs.core.windows.net/gold/fact_traffic_counts


## 7. Validation

In [0]:
from pyspark.sql import DataFrame

errors = []

def check_pk(df: DataFrame, cols: list, label: str):
    total  = df.count()
    unique = df.select(cols).distinct().count()
    if total != unique:
        errors.append(f"[FAIL] {label}: {total - unique:,} duplicate PKs on {cols}")
    else:
        print(f"[OK]   {label}: PK unique ({total:,} rows)")

def check_fk(fact_col: str, dim_df: DataFrame, dim_col: str, label: str):
    fact_df  = spark.read.format("delta").load(GOLD_FACT)
    # For nullable columns (segment_id), only check non-null values
    fact_vals = fact_df.filter(F.col(fact_col).isNotNull()).select(fact_col).distinct()
    dim_vals  = dim_df.select(dim_col).distinct()
    orphans   = fact_vals.join(dim_vals, fact_vals[fact_col] == dim_vals[dim_col], "left_anti")
    n = orphans.count()
    if n:
        errors.append(f"[FAIL] {label}: {n:,} orphan FK values")
    else:
        print(f"[OK]   {label}: no orphan FKs")

# Reload Gold tables
g_segment  = spark.read.format("delta").load(GOLD_SEGMENT)
g_location = spark.read.format("delta").load(GOLD_LOCATION)
g_time     = spark.read.format("delta").load(GOLD_TIME)
g_fact     = spark.read.format("delta").load(GOLD_FACT)

# PK uniqueness
check_pk(g_segment,  ["segment_id"],                          "dim_road_segment")
check_pk(g_location, ["site_id"],                             "dim_location")
check_pk(g_time,     ["date_id"],                             "dim_time")
check_pk(g_fact,     ["date_id", "site_id", "vehicle_class"], "fact_traffic_counts")

# Referential integrity
check_fk("date_id",    g_time,     "date_id",    "fact → dim_time")
check_fk("site_id",    g_location, "site_id",    "fact → dim_location")
check_fk("segment_id", g_segment,  "segment_id", "fact → dim_road_segment (non-null only)")

# Null check on measure
null_n = g_fact.filter(F.col("daily_count").isNull()).count()
total  = g_fact.count()
pct    = null_n / total * 100
if pct > 5:
    errors.append(f"[WARN] daily_count: {null_n:,} nulls ({pct:.1f}%)")
else:
    print(f"[OK]   daily_count nulls: {null_n:,} ({pct:.2f}%)")

# Summary stats
print(f"\nDate range : {date_bounds.min_date} → {date_bounds.max_date}")
print(f"Distinct sites in fact   : {g_fact.select('site_id').distinct().count():,}")
print(f"Sites matched to segment : "
      f"{g_location.filter(F.col('segment_id').isNotNull()).count()} / {g_location.count()}")

print()
if errors:
    for e in errors:
        print(e)
else:
    print("All validation checks passed.")

[OK]   dim_road_segment: PK unique (25,366 rows)
[OK]   dim_location: PK unique (233 rows)
[OK]   dim_time: PK unique (2,072 rows)
[OK]   fact_traffic_counts: PK unique (353,126 rows)
[OK]   fact → dim_time: no orphan FKs
[OK]   fact → dim_location: no orphan FKs
[OK]   fact → dim_road_segment (non-null only): no orphan FKs
[OK]   daily_count nulls: 0 (0.00%)

Date range : 2018-01-01 → 2023-09-03
Distinct sites in fact   : 233
Sites matched to segment : 233 / 233

All validation checks passed.


## 8. Register Gold tables in Spark catalog

In [0]:
spark.sql("CREATE DATABASE IF NOT EXISTS road_pipeline")

tables = {
    "dim_road_segment"   : GOLD_SEGMENT,
    "dim_location"       : GOLD_LOCATION,
    "dim_time"           : GOLD_TIME,
    "fact_traffic_counts": GOLD_FACT,
}

for table_name, path in tables.items():
    spark.read.format("delta").load(path) \
        .write.format("delta") \
        .mode("overwrite") \
        .saveAsTable(f"road_pipeline.{table_name}")
    print(f"Registered: road_pipeline.{table_name}")

print("\nStar schema ready. Example queries:")
print("  SELECT * FROM road_pipeline.fact_traffic_counts LIMIT 10")
print("  SELECT * FROM road_pipeline.dim_road_segment WHERE road_class = 'arterial'")

Registered: road_pipeline.dim_road_segment
Registered: road_pipeline.dim_location
Registered: road_pipeline.dim_time
Registered: road_pipeline.fact_traffic_counts

Star schema ready. Example queries:
  SELECT * FROM road_pipeline.fact_traffic_counts LIMIT 10
  SELECT * FROM road_pipeline.dim_road_segment WHERE road_class = 'arterial'
